# 单细胞特征选择

按照 [sc-best-practices](https://www.sc-best-practices.org/preprocessing_visualization/feature_selection.html) 使用 binomial deviance 选择基因。

运行前请把代码中的 `path/to/...` 改成自己的数据路径，并从项目根目录运行。

## 1. 读取 raw counts

deviance 使用 `adata.X` 中的原始整数 count。

In [ ]:
from pathlib import Path

import anndata as ad
import numpy as np

project_dir = Path(".")
input_path = Path("path/to/pbmc_1k_normalized.h5ad")
adata = ad.read_h5ad(input_path)
raw_counts = adata.X
print(adata)

## 2. 过滤低频基因

保留至少在 20 个细胞中检测到的基因。

In [ ]:
detected_cells = np.asarray((raw_counts > 0).sum(axis=0)).ravel()
keep_genes = detected_cells >= 20
adata_fs = adata[:, keep_genes].copy()

print("原始基因数:", adata.n_vars)
print("保留基因数:", adata_fs.n_vars)

## 3. 用 R 计算 binomial deviance

矩阵转为 gene × cell 后交给 `scry`。

In [ ]:
import shutil
import subprocess
import tempfile

import pandas as pd
from scipy import io

gene_symbol = adata_fs.var.get(
    "gene_symbol",
    pd.Series(adata_fs.var_names, index=adata_fs.var_names),
).astype(str)

tmp = Path(tempfile.mkdtemp(prefix="pbmc_1k_deviance_"))
counts_path = tmp / "counts_gene_by_cell.mtx"
genes_path = tmp / "genes.tsv"
deviance_path = tmp / "deviance.tsv"

io.mmwrite(counts_path, adata_fs.X.tocsr().T.tocoo())
pd.DataFrame({
    "var_name": adata_fs.var_names.astype(str),
    "gene_symbol": gene_symbol.to_numpy(),
}).to_csv(genes_path, sep="\t", index=False)

rscript_path = shutil.which("Rscript")

run = subprocess.run(
    [
        str(rscript_path),
        str(project_dir / "scripts/run_deviance_feature_selection.R"),
        str(counts_path),
        str(genes_path),
        str(deviance_path),
    ],
    check=True,
)
print("deviance 计算完成")

## 4. 选择前 4000 个基因

In [ ]:
deviance = pd.read_csv(deviance_path, sep="\t")
binomial_deviance = deviance["binomial_deviance"].to_numpy(dtype=float)

idx = binomial_deviance.argsort()[-4000:]
mask = np.zeros(adata.var_names.shape, dtype=bool)
mask[adata.var_names.get_indexer(adata_fs.var_names[idx])] = True

adata.var["highly_deviant"] = mask
adata.var["binomial_deviance"] = np.nan
adata.var.loc[adata_fs.var_names, "binomial_deviance"] = binomial_deviance

print("选中的基因数:", int(mask.sum()))

## 5. 可视化

这里仅使用 scran-normalized layer 计算均值和离散度作图。

In [ ]:
import matplotlib.pyplot as plt
import scanpy as sc
import seaborn as sns

sc.pp.highly_variable_genes(adata, layer="scran_normalization")

ax = sns.scatterplot(
    data=adata.var,
    x="means",
    y="dispersions",
    hue="highly_deviant",
    s=5,
)
ax.set_xlim(None, 1.5)
ax.set_ylim(None, 3)
plt.show()

## 6. 保存结果

In [ ]:
output_path = project_dir / "results/pbmc_1k_feature_selection.h5ad"
output_path.parent.mkdir(exist_ok=True)
adata.write(output_path)
shutil.rmtree(tmp, ignore_errors=True)
print("已保存:", output_path)